In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


Dataset binario (toy)

In [ ]:
X, y = make_classification(
    n_samples=1200,
    n_features=10,
    n_informative=4,
    n_redundant=2,
    n_clusters_per_class=2,
    class_sep=1.0,
    flip_y=0.05,
    random_state=0
)

CV estratificada + métrica

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scoring = "roc_auc"

Pipeline + cross_val_score (BIEN)

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000))
])

scores_good = cross_val_score(pipe, X, y, cv=cv, scoring=scoring)

print("\nBIEN (Pipeline + CV)")
print("scores:", np.round(scores_good, 3))
print("media ± std:", round(scores_good.mean(), 3), "±", round(scores_good.std(), 3))


MAL: escalar ANTES de CV (leakage)

In [ ]:
scaler = StandardScaler()
X_bad = scaler.fit_transform(X)  # <-- MAL: usa todo X (incluye futuros test-folds)

clf = LogisticRegression(max_iter=2000)
scores_bad = cross_val_score(clf, X_bad, y, cv=cv, scoring=scoring)

print("\nMAL (escalado global + CV)")
print("scores:", np.round(scores_bad, 3))
print("media ± std:", round(scores_bad.mean(), 3), "±", round(scores_bad.std(), 3))

print("\nDiferencia de medias (mal - bien):", round(scores_bad.mean() - scores_good.mean(), 4))

Otro ejemplo: SelectKBest (leakage MUY grande)

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression

X, y = make_classification(
    n_samples=250,
    n_features=500,
    n_informative=10,
    n_redundant=40,
    class_sep=0.5,
    flip_y=0.2,
    weights=[0.75, 0.25],
    random_state=0
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scoring = "roc_auc"

# BIEN: selección supervisada dentro del Pipeline (en cada fold, solo con train)
pipe_good = Pipeline([
    ("scaler", StandardScaler()),
    ("sel", SelectKBest(score_func=f_classif, k=20)),
    ("clf", LogisticRegression(max_iter=2000))
])
scores_good = cross_val_score(pipe_good, X, y, cv=cv, scoring=scoring)

# MAL: seleccionar features con TODO el dataset (incluye los y del futuro test) -> leakage fuerte
sel = SelectKBest(score_func=f_classif, k=20)
X_sel_bad = sel.fit_transform(X, y)  # <-- MAL (usa y de todo el dataset)

scaler = StandardScaler()
X_bad = scaler.fit_transform(X_sel_bad)  # también MAL, pero aquí lo gordo ya fue SelectKBest

clf = LogisticRegression(max_iter=2000)
scores_bad = cross_val_score(clf, X_bad, y, cv=cv, scoring=scoring)

print("BIEN (Pipeline):", scores_good, "mean±std:", scores_good.mean(), scores_good.std())
print("MAL  (leakage) :", scores_bad,  "mean±std:", scores_bad.mean(),  scores_bad.std())
print("Diferencia media (mal-bien):", scores_bad.mean() - scores_good.mean())